<a href="https://colab.research.google.com/github/rafallex/A3-ADL/blob/main/improvedv13_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Cancer Classification Challenge 2026 — v13 (Colab edition)

Same strategy as `improvedv13.ipynb` (v11 settings + SSL rotation pretraining), adapted for Google Colab with **full resumability**:

- **JPEG cache is persisted to Drive** (`cache_train.pkl`, `cache_test.pkl`). Built once, reloaded in seconds on every subsequent session.
- **SSL backbone is persisted to Drive** (`ssl_backbone.pt`). If it exists, the SSL stage is skipped entirely.
- **Every CV checkpoint is persisted to Drive**. If `fold0_seed1_best.pt` already exists, that model is skipped.
- **Submission written to Drive** at the end.

**Why this matters on Colab:** sessions disconnect after ~12 h or if the browser closes too long. With persistence, you can rerun the notebook from top and it picks up exactly where it left off. A full run takes ~4 h on Colab's free T4; if it disconnects after 3 h, you reconnect and finish the last hour without losing anything.

## One-time setup

1. **Upload the dataset folder to Drive** so it lives at one of:
   - `/content/drive/MyDrive/multimodal-cancer-classification-challenge-2026/` containing `BF/`, `FL/`, `train.csv`, `sampleSubmission.csv`
   - Or any folder under `MyDrive` containing those four items — the notebook auto-discovers it.

2. **Runtime → Change runtime type → GPU** (T4 on free tier).

3. Run cells top to bottom.

In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Persistent project root on Drive — everything except the JPEG decode cache lives here.
import os
PROJECT_DIR = "/content/drive/MyDrive/a3-cancer-2026"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project dir:", PROJECT_DIR)

Mounted at /content/drive
Project dir: /content/drive/MyDrive/a3-cancer-2026


In [2]:
import os
os.environ["PYTHONUNBUFFERED"] = "1"

import re, io, json, time, random, glob, functools, gc, pickle
from pathlib import Path

print = functools.partial(print, flush=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from PIL import Image
import matplotlib.pyplot as plt

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available(),
      "n_gpu:", torch.cuda.device_count())
!nvidia-smi -L

torch: 2.10.0+cu128 cuda: True n_gpu: 1
GPU 0: Tesla T4 (UUID: GPU-6765f206-2fda-9d3b-6ee3-8f5715dd0e5a)


In [3]:
# Locate the dataset. Tries, in order:
#   1. The cache pickles on Drive (don't need raw data if cache exists).
#   2. An already-unzipped folder under /content (fast local SSD).
#   3. An already-unzipped folder under MyDrive.
#   4. A zip file under MyDrive — auto-unzips to /content (~3-5 min).
#
# Unzipping to /content (local SSD) is ~10x faster than reading from Drive
# during the cache-build phase. The cache pickle on Drive means we only have
# to do this once per Drive account anyway.
import shutil, subprocess, glob as _glob
DATASET_NAME = "multimodal-cancer-classification-challenge-2026"
LOCAL_DATA_DIR = Path("/content") / DATASET_NAME

DATA_ROOT = None

# Step 1: cache pickles already exist? Skip raw data entirely.
PROJECT_DIR_P = Path(PROJECT_DIR)
cache_train = PROJECT_DIR_P / "cache" / "cache_train.pkl"
cache_test  = PROJECT_DIR_P / "cache" / "cache_test.pkl"
if cache_train.exists() and cache_test.exists():
    print("Cache pickles already on Drive — raw data not needed, will load from cache.")
    DATA_ROOT = LOCAL_DATA_DIR  # placeholder; not actually read from
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    # Touch dummy files so subsequent path checks don't error — they're never read.
    for f in ("train.csv", "sampleSubmission.csv"):
        (LOCAL_DATA_DIR / f).touch(exist_ok=True)
    print("DATA_ROOT =", DATA_ROOT, "(cache-only mode)")

# Step 2: local unzipped folder
if DATA_ROOT is None and (LOCAL_DATA_DIR / "train.csv").exists():
    DATA_ROOT = LOCAL_DATA_DIR
    print(f"Found unzipped dataset locally at {DATA_ROOT}")

# Step 3: unzipped folder on Drive
if DATA_ROOT is None:
    drive_candidates = list(Path("/content/drive/MyDrive").glob(f"**/{DATASET_NAME}"))
    for c in drive_candidates:
        if (c / "train.csv").exists():
            DATA_ROOT = c
            print(f"Found unzipped dataset on Drive at {DATA_ROOT} (slow — consider unzipping locally)")
            break

# Step 4: zip on Drive — unzip to /content/
if DATA_ROOT is None:
    zip_candidates = _glob.glob(f"/content/drive/MyDrive/**/{DATASET_NAME}.zip", recursive=True)
    if not zip_candidates:
        raise FileNotFoundError(
            f"No dataset found. Looked for:\n"
            f"  - cache pickles in {PROJECT_DIR_P/'cache'}\n"
            f"  - unzipped folder at {LOCAL_DATA_DIR}\n"
            f"  - unzipped folder under MyDrive\n"
            f"  - {DATASET_NAME}.zip under MyDrive\n"
            f"Please upload the dataset.")
    zip_path = zip_candidates[0]
    print(f"Found zip: {zip_path}")
    print(f"Unzipping to /content/ (this is fast, ~3-5 min)...")
    t0 = time.time()
    subprocess.run(["unzip", "-q", zip_path, "-d", "/content/"], check=True)
    print(f"  unzipped in {time.time()-t0:.1f}s")
    if (LOCAL_DATA_DIR / "train.csv").exists():
        DATA_ROOT = LOCAL_DATA_DIR
    else:
        # The zip might have a different top-level layout; search /content for train.csv
        found = list(Path("/content").glob("**/train.csv"))
        for p in found:
            if all((p.parent / f).exists() for f in ("sampleSubmission.csv", "BF", "FL")):
                DATA_ROOT = p.parent; break
        if DATA_ROOT is None:
            raise FileNotFoundError(
                "Unzip succeeded but couldn't locate train.csv + BF + FL afterwards.")
        print(f"Dataset root after unzip: {DATA_ROOT}")

assert DATA_ROOT is not None, "Could not locate or extract dataset"
print("DATA_ROOT =", DATA_ROOT)
if (DATA_ROOT / "train.csv").stat().st_size > 0:
    print("Contents:", sorted(p.name for p in DATA_ROOT.iterdir()))

Found zip: /content/drive/MyDrive/Uppsala/Year 1/Period 4/Advanced Deep Learning/A3/multimodal-cancer-classification-challenge-2026.zip
Unzipping to /content/ (this is fast, ~3-5 min)...
  unzipped in 45.4s
Dataset root after unzip: /content
DATA_ROOT = /content
Contents: ['.config', 'BF', 'FL', 'sampleSubmission.csv', 'sample_data', 'train.csv']


In [4]:
PROJECT_DIR = Path(PROJECT_DIR)
OUT_DIR     = PROJECT_DIR / "runs"           # checkpoints + OOF csvs (on Drive)
CACHE_DIR   = PROJECT_DIR / "cache"           # cached JPEG bytes (on Drive)
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# CV
N_SPLITS    = 3
BASE_SEED   = 1
SEEDS       = [1, 2]

# Supervised optimization (v11 settings)
EPOCHS      = 10
PATIENCE    = 5
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP   = 1.0
MIXUP_ALPHA = 0.1
DROPOUT     = 0.3

# SSL pretraining
SSL_ENABLED  = True
SSL_EPOCHS   = 6
SSL_LR       = 1e-3
SSL_BATCH    = 384
SSL_CKPT     = OUT_DIR / "ssl_backbone.pt"

# Sampler
NUM_WORKERS = 2
PATIENTS_PER_BATCH = 4

# Full-data model
TRAIN_FULL_DATA_MODEL = True
FULL_DATA_CKPT        = OUT_DIR / "fulldata_best.pt"

# Ensemble gate
ENSEMBLE_MIN_AUC = 0.78

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Dataset-level normalization (v11 stats)
BF_MEAN, BF_STD = 0.504, 0.216
FL_MEAN, FL_STD = 0.100, 0.144

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(BASE_SEED)

print(f"OUT_DIR   = {OUT_DIR}")
print(f"CACHE_DIR = {CACHE_DIR}")

OUT_DIR   = /content/drive/MyDrive/a3-cancer-2026/runs
CACHE_DIR = /content/drive/MyDrive/a3-cancer-2026/cache


In [5]:
PAT_RE = re.compile(r"^pat_(\d+)_image_\d+\.jpg$")

def parse_patient_id(filename):
    m = PAT_RE.match(Path(filename).name)
    return int(m.group(1)) if m else None

def load_train_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    df["patient_id"] = df["Name"].map(parse_patient_id).astype(int)
    return df

def load_test_df(path):
    df = pd.read_csv(path); df.columns = [c.strip() for c in df.columns]
    return df

def cache_split_from_disk(names, bf_dir, fl_dir, label=""):
    """Read JPEG bytes from Drive into two dicts (slow first time, fast after pickle)."""
    bf_dir, fl_dir = Path(bf_dir), Path(fl_dir)
    bf, fl = {}, {}
    t0 = time.time()
    for i, n in enumerate(names):
        with open(bf_dir / n, "rb") as f: bf[n] = f.read()
        with open(fl_dir / n, "rb") as f: fl[n] = f.read()
        if (i + 1) % 10000 == 0:
            print(f"  [{label}] read {i+1}/{len(names)} in {time.time()-t0:.1f}s")
    print(f"  [{label}] read {len(names)} in {time.time()-t0:.1f}s")
    return bf, fl

def load_or_build_cache(split, names, bf_dir, fl_dir):
    """Pickle-backed cache so we only pay the slow read once."""
    pkl = CACHE_DIR / f"cache_{split}.pkl"
    if pkl.exists():
        t0 = time.time()
        with open(pkl, "rb") as f:
            data = pickle.load(f)
        print(f"  [{split}] loaded cache from disk in {time.time()-t0:.1f}s ({len(data['bf'])} items)")
        return data["bf"], data["fl"]
    bf, fl = cache_split_from_disk(names, bf_dir, fl_dir, label=split)
    t0 = time.time()
    with open(pkl, "wb") as f:
        pickle.dump({"bf": bf, "fl": fl}, f, protocol=pickle.HIGHEST_PROTOCOL)
    print(f"  [{split}] wrote cache to {pkl} in {time.time()-t0:.1f}s")
    return bf, fl

class CachedCellDataset(Dataset):
    def __init__(self, df, bf_cache, fl_cache, bf_tf, fl_tf, paired_tf=None):
        self.df = df.reset_index(drop=True)
        self.bf_cache = bf_cache; self.fl_cache = fl_cache
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
        self.paired_tf = paired_tf
    def __len__(self): return len(self.df)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        row = self.df.iloc[idx]; name = row["Name"]
        bf = self.bf_tf(self._decode(self.bf_cache[name]))
        fl = self.fl_tf(self._decode(self.fl_cache[name]))
        if self.paired_tf is not None:
            bf, fl = self.paired_tf(bf, fl)
        label = int(row["Diagnosis"]) if "Diagnosis" in row else -1
        return {"bf": bf, "fl": fl, "label": label, "name": name}

class SSLRotationDataset(Dataset):
    def __init__(self, all_pairs, bf_caches, fl_caches, bf_tf, fl_tf):
        self.pairs = all_pairs
        self.bf_caches = bf_caches
        self.fl_caches = fl_caches
        self.bf_tf = bf_tf; self.fl_tf = fl_tf
    def __len__(self): return len(self.pairs)
    @staticmethod
    def _decode(buf): return Image.open(io.BytesIO(buf)).convert("L")
    def __getitem__(self, idx):
        name, split = self.pairs[idx]
        bf = self.bf_tf(self._decode(self.bf_caches[split][name]))
        fl = self.fl_tf(self._decode(self.fl_caches[split][name]))
        k = random.randint(0, 3)
        if k:
            bf = torch.rot90(bf, k, dims=(-2, -1))
            fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < 0.5:
            bf, fl = TF.hflip(bf), TF.hflip(fl)
        return {"bf": bf, "fl": fl, "rot": k}

In [6]:
def stratified_patient_kfold(df, n_splits=3, seed=1):
    skgf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    y = df["Diagnosis"].to_numpy(); groups = df["patient_id"].to_numpy()
    splits = list(skgf.split(df, y=y, groups=groups))
    for f, (_, va) in enumerate(splits):
        if len(np.unique(y[va])) < 2:
            raise ValueError(f"Fold {f} has only one class — try another seed.")
    return splits

def summarize_split(df, tr, va):
    trd, vad = df.iloc[tr], df.iloc[va]
    return (f"train: {len(tr):>6} cells, {trd['patient_id'].nunique():>2}p, "
            f"pos {trd['Diagnosis'].mean():.3f} | "
            f"val: {len(va):>6} cells, {vad['patient_id'].nunique():>2}p, "
            f"pos {vad['Diagnosis'].mean():.3f} | "
            f"val pats: {sorted(vad['patient_id'].unique().tolist())}")

class PatientBalancedSampler(Sampler):
    def __init__(self, df, batch_size, patients_per_batch=4, seed=0):
        assert batch_size % patients_per_batch == 0
        self.df = df.reset_index(drop=True)
        self.batch_size = batch_size
        self.per_pat = batch_size // patients_per_batch
        self.patients_per_batch = patients_per_batch
        self.rng = np.random.default_rng(seed)
        self.by_pat = {p: np.array(g.index.tolist())
                       for p, g in self.df.groupby("patient_id")}
        self.patients = list(self.by_pat.keys())
        self.epoch_len = len(self.df) // batch_size * batch_size
    def __len__(self): return self.epoch_len
    def __iter__(self):
        out = []
        for _ in range(self.epoch_len // self.batch_size):
            pats = self.rng.choice(self.patients,
                                   size=min(self.patients_per_batch, len(self.patients)),
                                   replace=False)
            for p in pats:
                idxs = self.by_pat[p]
                out.extend(self.rng.choice(idxs, size=self.per_pat,
                                           replace=len(idxs) < self.per_pat).tolist())
        return iter(out)

def _to_tensor_norm(mean, std):
    def fn(img):
        t = TF.to_tensor(img)
        return TF.normalize(t, [mean], [std])
    return fn
to_tensor_bf = _to_tensor_norm(BF_MEAN, BF_STD)
to_tensor_fl = _to_tensor_norm(FL_MEAN, FL_STD)

class PairedGeoAug:
    def __init__(self, p_hflip=0.5, p_vflip=0.5, rot90=True, max_rot=10.0):
        self.p_hflip = p_hflip; self.p_vflip = p_vflip
        self.rot90 = rot90; self.max_rot = max_rot
    def __call__(self, bf, fl):
        if self.rot90:
            k = random.randint(0, 3)
            if k:
                bf = torch.rot90(bf, k, dims=(-2, -1))
                fl = torch.rot90(fl, k, dims=(-2, -1))
        if random.random() < self.p_hflip: bf, fl = TF.hflip(bf), TF.hflip(fl)
        if random.random() < self.p_vflip: bf, fl = TF.vflip(bf), TF.vflip(fl)
        if self.max_rot > 0:
            a = random.uniform(-self.max_rot, self.max_rot)
            bf, fl = TF.rotate(bf, a), TF.rotate(fl, a)
        return bf, fl

def train_modality_transform(modality):
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])
def eval_modality_transform(modality):
    return to_tensor_bf if modality == "bf" else to_tensor_fl
def ssl_modality_transform(modality):
    norm = to_tensor_bf if modality == "bf" else to_tensor_fl
    return T.Compose([T.ColorJitter(brightness=0.2, contrast=0.2), norm])

In [7]:
def _make_resnet18_branch(pretrained=True):
    weights = "DEFAULT" if pretrained else None
    net = models.resnet18(weights=weights)
    w = net.conv1.weight.data
    new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    if pretrained:
        new_conv.weight.data = w.mean(dim=1, keepdim=True)
    net.conv1 = new_conv
    fd = net.fc.in_features; net.fc = nn.Identity()
    return net, fd  # 512

class MultimodalClassifier(nn.Module):
    def __init__(self, pretrained=True, dropout=DROPOUT):
        super().__init__()
        self.bf_branch, fd = _make_resnet18_branch(pretrained)
        self.fl_branch, _  = _make_resnet18_branch(pretrained)
        self.head = nn.Sequential(
            nn.Linear(fd * 2, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.head(feat).squeeze(-1)

class SSLRotationModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.bf_branch, fd = _make_resnet18_branch(pretrained)
        self.fl_branch, _  = _make_resnet18_branch(pretrained)
        self.rot_head = nn.Sequential(
            nn.Linear(fd * 2, 256), nn.BatchNorm1d(256),
            nn.ReLU(inplace=True), nn.Linear(256, 4))
    def forward(self, bf, fl):
        feat = torch.cat([self.bf_branch(bf), self.fl_branch(fl)], dim=1)
        return self.rot_head(feat)

In [8]:
df_train = load_train_df(DATA_ROOT / "train.csv")
df_test  = load_test_df(DATA_ROOT / "sampleSubmission.csv")
print(f"Train: {len(df_train)} cells, {df_train['patient_id'].nunique()} patients, "
      f"pos rate {df_train['Diagnosis'].mean():.4f}")
print(f"Test:  {len(df_test)} cells")

print("\n--- Caching JPEG bytes (will use Drive-side pickle if present) ---")
bf_train_cache, fl_train_cache = load_or_build_cache(
    "train", df_train["Name"].tolist(),
    DATA_ROOT / "BF" / "train", DATA_ROOT / "FL" / "train")
bf_test_cache, fl_test_cache = load_or_build_cache(
    "test", df_test["Name"].tolist(),
    DATA_ROOT / "BF" / "test", DATA_ROOT / "FL" / "test")

approx_mb = (sum(len(b) for b in bf_train_cache.values()) +
             sum(len(b) for b in fl_train_cache.values()) +
             sum(len(b) for b in bf_test_cache.values()) +
             sum(len(b) for b in fl_test_cache.values())) / (1024 * 1024)
print(f"\nApprox RAM used by JPEG cache: {approx_mb:.0f} MB")

Train: 114302 cells, 12 patients, pos rate 0.3876
Test:  59040 cells

--- Caching JPEG bytes (will use Drive-side pickle if present) ---
  [train] read 10000/114302 in 0.6s
  [train] read 20000/114302 in 1.2s
  [train] read 30000/114302 in 2.3s
  [train] read 40000/114302 in 3.5s
  [train] read 50000/114302 in 4.7s
  [train] read 60000/114302 in 6.0s
  [train] read 70000/114302 in 6.9s
  [train] read 80000/114302 in 7.6s
  [train] read 90000/114302 in 8.2s
  [train] read 100000/114302 in 8.8s
  [train] read 110000/114302 in 9.6s
  [train] read 114302 in 9.9s
  [train] wrote cache to /content/drive/MyDrive/a3-cancer-2026/cache/cache_train.pkl in 2.9s
  [test] read 10000/59040 in 0.6s
  [test] read 20000/59040 in 1.2s
  [test] read 30000/59040 in 1.8s
  [test] read 40000/59040 in 2.5s
  [test] read 50000/59040 in 3.0s
  [test] read 59040 in 3.6s
  [test] wrote cache to /content/drive/MyDrive/a3-cancer-2026/cache/cache_test.pkl in 0.5s

Approx RAM used by JPEG cache: 755 MB


In [ ]:
def train_ssl_rotation(epochs=SSL_EPOCHS, batch_size=SSL_BATCH, lr=SSL_LR):
    seed_everything(BASE_SEED + 7)
    all_pairs = ([(n, "train") for n in df_train["Name"].tolist()] +
                 [(n, "test")  for n in df_test["Name"].tolist()])
    print(f"SSL on {len(all_pairs)} cell pairs")
    random.shuffle(all_pairs)
    bf_caches = {"train": bf_train_cache, "test": bf_test_cache}
    fl_caches = {"train": fl_train_cache, "test": fl_test_cache}
    ds = SSLRotationDataset(all_pairs, bf_caches, fl_caches,
                            ssl_modality_transform("bf"), ssl_modality_transform("fl"))
    loader = DataLoader(ds, batch_size=batch_size, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    model = SSLRotationModel(pretrained=True).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(loader),
        epochs=epochs, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None
    for ep in range(epochs):
        t0 = time.time(); losses, correct, total = [], 0, 0
        model.train()
        for i, batch in enumerate(loader):
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            rot = batch["rot"].to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=scaler is not None):
                logits = model(bf, fl); loss = criterion(logits, rot)
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                old_scale = scaler.get_scale(); scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= old_scale: sched.step()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); sched.step()
            losses.append(loss.item())
            correct += (logits.argmax(1) == rot).sum().item(); total += rot.size(0)
            if (i + 1) % 100 == 0:
                print(f"    step {i+1}/{len(loader)} | loss {np.mean(losses[-100:]):.4f} | acc {correct/total:.3f}")
        print(f"  SSL ep {ep} | loss {np.mean(losses):.4f} | acc {correct/total:.3f} | {time.time()-t0:.1f}s")
    torch.save({"bf_branch": model.bf_branch.state_dict(),
                "fl_branch": model.fl_branch.state_dict()}, SSL_CKPT)
    print(f"Saved SSL backbone to {SSL_CKPT}")
    del model, optimizer, sched, scaler, loader, ds
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if SSL_ENABLED and not SSL_CKPT.exists():
    print("=== Stage 1: SSL rotation pretraining ===")
    train_ssl_rotation()
elif SSL_CKPT.exists():
    print(f"SSL backbone already exists at {SSL_CKPT} — skipping pretraining.")
else:
    print("(SSL disabled — will use ImageNet init)")

=== Stage 1: SSL rotation pretraining ===
SSL on 173342 cell pairs
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 109MB/s]


    step 100/451 | loss 1.2889 | acc 0.365
    step 200/451 | loss 0.8741 | acc 0.488
    step 300/451 | loss 0.7537 | acc 0.545
    step 400/451 | loss 0.6700 | acc 0.580
  SSL ep 0 | loss 0.8692 | acc 0.592 | 370.3s
    step 100/451 | loss 0.6089 | acc 0.705
    step 200/451 | loss 0.5787 | acc 0.710
    step 300/451 | loss 0.5616 | acc 0.715
    step 400/451 | loss 0.5049 | acc 0.727
  SSL ep 1 | loss 0.5556 | acc 0.732 | 368.2s


In [ ]:
def load_ssl_branches(model, ssl_ckpt_path=SSL_CKPT):
    if not Path(ssl_ckpt_path).exists():
        print(f"  (no SSL ckpt at {ssl_ckpt_path}, keeping ImageNet init)")
        return
    state = torch.load(ssl_ckpt_path, map_location="cpu", weights_only=False)
    model.bf_branch.load_state_dict(state["bf_branch"], strict=False)
    model.fl_branch.load_state_dict(state["fl_branch"], strict=False)
    print("  loaded SSL backbone")

def mixup_batch(bf, fl, y, alpha=0.1):
    if alpha <= 0: return bf, fl, y
    lam = float(np.random.beta(alpha, alpha))
    idx = torch.randperm(bf.size(0), device=bf.device)
    return (lam * bf + (1 - lam) * bf[idx],
            lam * fl + (1 - lam) * fl[idx],
            lam * y  + (1 - lam) * y[idx])

def run_epoch(model, loader, optimizer, scaler, criterion, train,
              mixup_alpha=0.0, grad_clip=0.0, sched=None, log_every=0):
    model.train(train)
    losses, hard_ys, ps = [], [], []
    t_last = time.time()
    for i, batch in enumerate(loader):
        bf = batch["bf"].to(DEVICE, non_blocking=True)
        fl = batch["fl"].to(DEVICE, non_blocking=True)
        y  = batch["label"].float().to(DEVICE, non_blocking=True)
        hard_ys.append(batch["label"].numpy())
        if train and mixup_alpha > 0:
            bf, fl, y = mixup_batch(bf, fl, y, mixup_alpha)
        with torch.amp.autocast("cuda", enabled=scaler is not None):
            logits = model(bf, fl); loss = criterion(logits, y)
        if train:
            optimizer.zero_grad(set_to_none=True)
            if scaler is not None:
                scaler.scale(loss).backward()
                if grad_clip > 0:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if sched is not None and scaler.get_scale() >= old_scale:
                    sched.step()
            else:
                loss.backward()
                if grad_clip > 0: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()
                if sched is not None: sched.step()
        losses.append(loss.item())
        ps.append(torch.sigmoid(logits).detach().float().cpu().numpy())
        if log_every and (i + 1) % log_every == 0:
            dt = time.time() - t_last; t_last = time.time()
            print(f"    step {i+1}/{len(loader)} | {dt:.1f}s | loss {float(np.mean(losses[-log_every:])):.4f}")
    hard_ys = np.concatenate(hard_ys); ps = np.concatenate(ps)
    auc = roc_auc_score(hard_ys, ps) if len(np.unique(hard_ys)) > 1 else float("nan")
    return float(np.mean(losses)), auc, hard_ys, ps

def train_one_model(train_df, val_df, ckpt_path, oof_path, hist_path,
                   epochs, seed, sampler_kind="patient", track_oof=True):
    if Path(ckpt_path).exists():
        # Skip — already trained in a previous session.
        print(f"  (ckpt {Path(ckpt_path).name} exists, skipping)")
        if Path(hist_path).exists():
            h = json.load(open(hist_path))
            return h.get("best_auc", -1.0), h.get("best_ep", 0)
        return -1.0, 0

    seed_everything(seed)
    train_ds = CachedCellDataset(train_df, bf_train_cache, fl_train_cache,
                                 train_modality_transform("bf"),
                                 train_modality_transform("fl"),
                                 paired_tf=PairedGeoAug())
    if sampler_kind == "patient":
        sampler = PatientBalancedSampler(train_df, batch_size=BATCH_SIZE,
                                         patients_per_batch=PATIENTS_PER_BATCH, seed=seed)
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    val_loader = None
    if val_df is not None:
        val_ds = CachedCellDataset(val_df, bf_train_cache, fl_train_cache,
                                   eval_modality_transform("bf"),
                                   eval_modality_transform("fl"))
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

    model = MultimodalClassifier(pretrained=True, dropout=DROPOUT).to(DEVICE)
    if SSL_ENABLED: load_ssl_branches(model)

    pos = (train_df["Diagnosis"] == 1).sum()
    neg = (train_df["Diagnosis"] == 0).sum()
    pos_weight = torch.tensor(neg / max(pos, 1), device=DEVICE)
    print(f"  pos_weight={pos_weight.item():.3f}  seed={seed}  epochs={epochs}")
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR,
        steps_per_epoch=len(train_loader), epochs=epochs, pct_start=0.1)
    scaler = torch.amp.GradScaler("cuda") if DEVICE == "cuda" else None

    history, best_auc, best_ep, no_improve = [], -1.0, 0, 0
    for ep in range(epochs):
        t0 = time.time()
        tr_loss, tr_auc, _, _ = run_epoch(
            model, train_loader, optimizer, scaler, criterion, True,
            mixup_alpha=MIXUP_ALPHA, grad_clip=GRAD_CLIP, sched=sched, log_every=200)
        va_loss = va_auc = float("nan"); vy = vp = None
        if val_loader is not None:
            with torch.no_grad():
                va_loss, va_auc, vy, vp = run_epoch(
                    model, val_loader, None, None, criterion, False)
        dt = time.time() - t0
        print(f"  ep {ep:>2d} | tr_loss {tr_loss:.4f} tr_auc {tr_auc:.4f} "
              f"| va_loss {va_loss:.4f} va_auc {va_auc:.4f} | {dt:.1f}s")
        history.append({"epoch": ep, "tr_loss": tr_loss, "tr_auc": tr_auc,
                        "va_loss": va_loss, "va_auc": va_auc, "time": dt})
        save_now = False
        if val_loader is not None:
            if va_auc > best_auc:
                best_auc, best_ep, no_improve = va_auc, ep, 0; save_now = True
            else:
                no_improve += 1
        else:
            save_now = True; best_ep = ep
        if save_now:
            torch.save({"model": model.state_dict(), "epoch": ep,
                        "val_auc": va_auc if val_loader is not None else None,
                        "args": {"dropout": DROPOUT}}, ckpt_path)
            if track_oof and val_loader is not None:
                pd.DataFrame({"Name": val_df["Name"].values,
                              "patient_id": val_df["patient_id"].values,
                              "y_true": vy, "y_pred": vp}).to_csv(oof_path, index=False)
        if val_loader is not None and no_improve >= PATIENCE:
            print(f"  Early stopping at epoch {ep}"); break
    with open(hist_path, "w") as f:
        json.dump({"history": history, "best_auc": best_auc, "best_ep": best_ep}, f, indent=2)
    del model, optimizer, sched, scaler, train_loader
    if val_loader is not None: del val_loader
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return best_auc, best_ep

In [ ]:
print("=== Stage 2: Supervised training (SSL-initialized) ===")
splits = stratified_patient_kfold(df_train, N_SPLITS, seed=BASE_SEED)
print(f"CV folds={len(splits)}  seeds={SEEDS}\n")

all_results = []
for fold, (tr, va) in enumerate(splits):
    print(f"=== FOLD {fold} ===")
    print("  " + summarize_split(df_train, tr, va))
    train_df = df_train.iloc[tr].reset_index(drop=True)
    val_df   = df_train.iloc[va].reset_index(drop=True)
    for seed in SEEDS:
        tag = f"fold{fold}_seed{seed}"
        print(f"  --- {tag} ---")
        ckpt = OUT_DIR / f"{tag}_best.pt"
        oof  = OUT_DIR / f"{tag}_oof.csv"
        hist = OUT_DIR / f"{tag}_history.json"
        best_auc, best_ep = train_one_model(
            train_df, val_df, ckpt, oof, hist, epochs=EPOCHS, seed=seed)
        all_results.append({"fold": fold, "seed": seed,
                            "best_auc": best_auc, "best_ep": best_ep})
        print(f"  {tag}: best AUC = {best_auc:.4f} at ep {best_ep}\n")

df_results = pd.DataFrame(all_results)
print("\n=== CV summary ===")
print(df_results.to_string(index=False))
print(f"Mean best AUC: {df_results['best_auc'].mean():.4f}  std {df_results['best_auc'].std():.4f}")

In [ ]:
if TRAIN_FULL_DATA_MODEL:
    median_ep = int(df_results['best_ep'].median()) + 1
    full_epochs = max(median_ep, 5)
    print(f"\n=== FULL-DATA MODEL (epochs={full_epochs}) ===")
    train_one_model(df_train, None, FULL_DATA_CKPT, None,
                    OUT_DIR / "fulldata_history.json",
                    epochs=full_epochs, seed=BASE_SEED + 100,
                    sampler_kind="patient", track_oof=False)
    print("  full-data model done.")

In [ ]:
fold_oofs = []
for fold in range(N_SPLITS):
    seed_oofs = []
    for seed in SEEDS:
        p = OUT_DIR / f"fold{fold}_seed{seed}_oof.csv"
        if p.exists(): seed_oofs.append(pd.read_csv(p))
    if not seed_oofs: continue
    base = seed_oofs[0][["Name", "patient_id", "y_true"]].copy()
    base["y_pred"] = np.mean([d["y_pred"].values for d in seed_oofs], axis=0)
    fold_oofs.append(base)
if fold_oofs:
    oof = pd.concat(fold_oofs, ignore_index=True)
    cell_auc = roc_auc_score(oof["y_true"], oof["y_pred"])
    pp = oof.groupby("patient_id").agg(
        mean_pred=("y_pred", "mean"), median_pred=("y_pred", "median"),
        label=("y_true", "first")).sort_values("mean_pred")
    pat_auc = roc_auc_score(pp["label"], pp["mean_pred"])
    print("=== OOF (seed-averaged) ===")
    print(pp.to_string())
    print(f"\ncell-level OOF AUC: {cell_auc:.4f}    patient-level AUC: {pat_auc:.4f}")

In [ ]:
def _d4(bf, fl):
    for k in range(4):
        bfr = torch.rot90(bf, k, dims=(-2, -1))
        flr = torch.rot90(fl, k, dims=(-2, -1))
        yield bfr, flr
        yield TF.hflip(bfr), TF.hflip(flr)

def load_model_from_ckpt(ckpt_path):
    state = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    args = state.get("args", {})
    dropout = float(args.get("dropout", DROPOUT))
    model = MultimodalClassifier(pretrained=False, dropout=dropout).to(DEVICE)
    model.load_state_dict(state["model"]); model.eval()
    return model

def predict_one_ckpt(ckpt_path, loader, tta=True):
    model = load_model_from_ckpt(ckpt_path)
    preds = []
    n_aug = 8 if tta else 1
    with torch.no_grad():
        for batch in loader:
            bf = batch["bf"].to(DEVICE, non_blocking=True)
            fl = batch["fl"].to(DEVICE, non_blocking=True)
            p = None
            for bf_t, fl_t in (_d4(bf, fl) if tta else [(bf, fl)]):
                with torch.amp.autocast("cuda", enabled=DEVICE == "cuda"):
                    pi = torch.sigmoid(model(bf_t, fl_t)).float()
                p = pi if p is None else p + pi
            preds.append((p / n_aug).cpu().numpy())
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return np.concatenate(preds)

test_ds = CachedCellDataset(df_test, bf_test_cache, fl_test_cache,
                            eval_modality_transform("bf"),
                            eval_modality_transform("fl"))
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

ckpts_all = sorted(glob.glob(str(OUT_DIR / "fold*_best.pt")))
ckpts = []
for c in ckpts_all:
    s = torch.load(c, map_location="cpu", weights_only=False)
    va = s.get("val_auc")
    if va is not None and va >= ENSEMBLE_MIN_AUC:
        ckpts.append(c); print(f"  KEEP {Path(c).name}  val_auc={va:.4f}")
    else:
        print(f"  DROP {Path(c).name}  val_auc={va}")
if TRAIN_FULL_DATA_MODEL and FULL_DATA_CKPT.exists():
    ckpts.append(str(FULL_DATA_CKPT))
    print("  KEEP fulldata_best.pt (always included)")

if not ckpts:
    raise RuntimeError(f"No checkpoints passed the gate at {ENSEMBLE_MIN_AUC}")

all_preds = []
for c in ckpts:
    t0 = time.time()
    all_preds.append(predict_one_ckpt(c, test_loader, tta=True))
    print(f"  {Path(c).name} done in {time.time()-t0:.1f}s")
preds = np.mean(all_preds, axis=0)

sub = pd.DataFrame({"Name": df_test["Name"].values, "Diagnosis": preds})
out_path = PROJECT_DIR / "submission.csv"
sub.to_csv(out_path, index=False)
# Also drop a local copy for download convenience.
sub.to_csv("/content/submission.csv", index=False)
print(f"\nWrote {out_path}  (mean pred = {preds.mean():.3f}, "
      f"min {preds.min():.3f}, max {preds.max():.3f})")
print(sub.head())
!wc -l /content/submission.csv